# PBISC bulk-only generated files: structure inventory

What is inside the K = 1,000 and K = 4,000 generated h5ad files for Ye 2026 sepsis and Imperial ESKD COVID, plus the Harmony-integrated subsets.

In [ ]:
# File-structure inventory of PBISC bulk-only generated data (K=1,000 and K=4,000; sepsis and COVID)
# Reads only HDF5 headers / categoricals with h5py, so it runs on a small machine.
import h5py, json, numpy as np, zipfile
ROOT = r"F:\PBISC"   # change to your mount
FILES = {
 "sepsis_K1k_draw0": "bulkonly_lora5_20261006/ye2026_sepsis/lora5_bulkonly_ye2026_sepsis_draw0.h5ad",
 "sepsis_K4k_draw0": "bulkonly_lora5_k4000_20261006/ye2026_sepsis/lora5k4_bulkonly_ye2026_sepsis_draw0.h5ad",
 "covid_K1k_draw0":  "bulkonly_lora5_20261006/imperial_eskd/lora5_bulkonly_imperial_eskd_draw0.h5ad",
 "covid_K4k_draw0":  "bulkonly_lora5_k4000_20261006/imperial_eskd/lora5k4_bulkonly_imperial_eskd_draw0.h5ad",
 "sepsis_K1k_harmony": "bulkonly_lora5_20261006/analysis/harmony/sepsis_k1000/sepsis_k1000_harmony_clustering.h5ad",
 "sepsis_K4k_harmony": "bulkonly_lora5_k4000_20261006/analysis/harmony/sepsis_k4000/sepsis_k4000_harmony_clustering.h5ad",
 "covid_K1k_harmony":  "bulkonly_lora5_20261006/analysis/harmony/covid_k1000/covid_k1000_harmony_embedding.h5ad",
 "covid_K4k_harmony":  "bulkonly_lora5_k4000_20261006/analysis/harmony/covid_k4000/covid_k4000_harmony_embedding.h5ad",
}

In [ ]:
def desc(g, k):
    o = g[k]
    if isinstance(o, h5py.Dataset): return f"{o.shape} {o.dtype}"
    et = o.attrs.get("encoding-type", "group")
    if et in ("csr_matrix", "csc_matrix"):
        return f"{et} {tuple(int(x) for x in o.attrs['shape'])} {o['data'].dtype} nnz={o['data'].shape[0]:,}"
    if et == "categorical": return f"cat{o['categories'].shape[0]}"
    return et

def inspect(path):
    with h5py.File(path, "r") as f:
        print("X:", desc(f, "X"))
        for g in ["layers", "obsm", "obsp"]:
            if g in f and len(f[g]): print(g + ":", {k: desc(f[g], k) for k in f[g]})
        for g in ["obs", "var"]:
            print(g, "|", ", ".join(f"{c}[{desc(f[g], c)}]" for c in f[g].keys()))
        print("uns:", list(f["uns"].keys()))
        print("measured_in_cohort:", int(f["var"]["measured_in_cohort"][:].sum()))

def depth(path, n=20000, seed=0):
    with h5py.File(path, "r") as f:
        ip = f["X"]["indptr"][:]; d = f["X"]["data"]
        idx = np.sort(np.random.default_rng(seed).choice(len(ip) - 1, n, replace=False))
        s = np.array([d[ip[i]:ip[i+1]].sum() for i in idx]); g = np.diff(ip)[idx]
    return np.median(s), np.percentile(s, [25, 75]), np.median(g)

for name, rel in FILES.items():
    print("#####", name); inspect(f"{ROOT}/{rel}")
for name in ["sepsis_K1k_draw0", "sepsis_K4k_draw0", "covid_K1k_draw0", "covid_K4k_draw0"]:
    print(name, "UMI median / IQR / genes median:", depth(f"{ROOT}/{FILES[name]}"))

In [ ]:
# Provenance of one draw file and the raw sampler NPZ members (zip listing only; no array is loaded)
with h5py.File(f"{ROOT}/{FILES['sepsis_K1k_draw0']}", "r") as f:
    prov = json.loads(f["uns"]["provenance"][()])
print({k: (v if len(str(v)) < 120 else str(v)[:120] + "...") for k, v in prov.items()})
z = zipfile.ZipFile(f"{ROOT}/bulkonly_lora5_20261006/raw_npz/ye2026_sepsis/draw0/lora5_bulkonly.npz")
for i in z.infolist(): print(i.filename, f"{i.file_size/1e6:.1f} MB")

## Recorded result (run 2026-10-18 on the PC workspace with h5py 3.16)

| file set | cells × genes per file | X | median UMI (IQR) | median genes/cell |
|---|---|---|---|---|
| sepsis K1k draw files (×5) | 183,000 × 6,000 (122 samples × 1,500) | int32 CSR counts | 1,652 (1,113–2,310) | 282 |
| sepsis K4k draw files (×5) | 488,000 × 6,000 (122 × 4,000) | int32 CSR counts | 1,610 (1,098–2,251) | 284 |
| COVID K1k draw files (×5) | 246,000 × 6,000 (164 × 1,500) | int32 CSR counts | 1,208 (871–1,650) | 256 |
| COVID K4k draw files (×5) | 656,000 × 6,000 (164 × 4,000) | int32 CSR counts | 1,188 (856–1,619) | 254 |
| sepsis K1k / K4k Harmony | 305,000 × 5,975 | float32 log1p CP10k; layers['counts'] int32 | — | — |
| COVID K1k / K4k Harmony | 410,000 × 5,927 | float32 log1p CP10k; layers['counts'] int32 | — | — |

Depth = 20,000 random cells of draw 0. Harmony files: obsm X_pca / X_pca_harmony (50 PCs; sepsis also X_umap), obsp neighbour graph; sepsis *_clustering.h5ad adds leiden_harmony10_r0.5 (12) / r1 (22) on the first 10 Harmony PCs. Batch covariate = draw; 500 cells per sample per draw; seed 20261007; HVG seurat 2,000.